In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import kagglehub
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, Input, Dropout

In [ ]:
path = kagglehub.dataset_download("asdasdasasdas/garbage-classification")

print("Path to dataset files:", path)

In [ ]:
import os

def find_image_class_root(base_path):
    """
    Walk the downloaded dataset and return the first directory whose
    immediate children are themselves directories containing image files
    (i.e. the true class-folder root), handling any extra nesting
    introduced by kagglehub's extraction (e.g. a duplicated
    'Garbage classification/Garbage classification/' wrapper).
    """
    for root, dirs, files in os.walk(base_path):
        if not dirs:
            continue
        # Check whether the subdirectories actually contain images
        sample_dir = os.path.join(root, dirs[0])
        sample_files = os.listdir(sample_dir)
        if any(f.lower().endswith(('.jpg', '.jpeg', '.png')) for f in sample_files):
            return root
    return base_path

dataset_path = find_image_class_root(path)
print("Using dataset path:", dataset_path)
print("Detected classes:", sorted(os.listdir(dataset_path)))

In [ ]:
from tensorflow.keras.utils import image_dataset_from_directory

# Load Dataset
train_ds = image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

val_ds = image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(128, 128),
    batch_size=32
)

In [ ]:
# Class Names
classes = train_ds.class_names

print("Classes:", classes)

In [ ]:
# Normalize Images
train_ds = train_ds.map(lambda x, y: (x / 255.0, y))
val_ds = val_ds.map(lambda x, y: (x / 255.0, y))

In [ ]:
# CNN Model
model = Sequential()

model.add(Input(shape=(128, 128, 3)))

model.add(Conv2D(32, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))

model.add(Flatten())

model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(len(classes), activation='softmax'))

In [ ]:
# Compile Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# Summary
model.summary()

In [ ]:
# Train Model
history = model.fit(
    train_ds,
    epochs=5,
    validation_data=val_ds
)

In [ ]:
# Evaluate Model
loss, accuracy = model.evaluate(val_ds)

print("Accuracy:", accuracy)

In [ ]:
# Predict First 5 Images
for image_batch, label_batch in val_ds.take(1):

    predictions = model.predict(image_batch[:5])

    for i in range(5):

        predicted_label = np.argmax(predictions[i])
        actual_label = label_batch[i].numpy()

        print(f"\nImage {i+1}")
        print(f"Actual    : {classes[actual_label]}")
        print(f"Predicted : {classes[predicted_label]}")

        plt.imshow(image_batch[i].numpy())
        plt.title(
            f"Actual: {classes[actual_label]}\n"
            f"Predicted: {classes[predicted_label]}"
        )
        plt.axis("off")
        plt.show()